In [23]:

from pathlib import Path
import os

# Main project folder (parent of Notebooks)
PROJECT_ROOT = Path.cwd().parent

# Set working directory to project root
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)


Project root: c:\Users\aravi\Documents\RAG Azure document Assistant


In [13]:
!pip install requests beautifulsoup4 tqdm


Defaulting to user installation because normal site-packages is not writeable


In [14]:
import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm

In [24]:
AZURE_SERVICES = {
    "Azure_Communication_Services":
    "https://learn.microsoft.com/en-us/azure/communication-services/"
}

SERVICE_NAME = "Azure_Communication_Services"
SERVICE_URL = AZURE_SERVICES[SERVICE_NAME]

DOCS_FOLDER = os.path.join("docs", SERVICE_NAME)
os.makedirs(DOCS_FOLDER, exist_ok=True)

In [25]:
def get_links():
    response = requests.get(SERVICE_URL)
    soup = BeautifulSoup(response.text,"html.parser")

    links=set()

    for a in soup.find_all("a",href=True):
        full=urljoin(SERVICE_URL,a["href"]).split("#")[0].split("?")[0]
        if full.startswith(SERVICE_URL):
            links.add(full)

    return sorted(list(links))

In [26]:
def get_markdown_url(page_url):
    try:
        response=requests.get(page_url,timeout=15)
        response.raise_for_status()

        soup=BeautifulSoup(response.text,"html.parser")
        meta=soup.find("meta",attrs={"name":"markdown_url"})

        if meta:
            return meta["content"]

    except Exception as e:
        print(e)

    return None

In [27]:
def download_markdown(page_url):
    md=get_markdown_url(page_url)

    if md is None:
        return None

    r=requests.get(md)
    r.raise_for_status()
    return r.text

In [28]:
def clean_markdown(md):

    if md.startswith("---"):
        parts=md.split("---",2)
        if len(parts)==3:
            md=parts[2]

    lines=[l.rstrip() for l in md.splitlines()]

    cleaned=[]
    blank=False

    for line in lines:
        if line.strip()=="":
            if not blank:
                cleaned.append("")
            blank=True
        else:
            cleaned.append(line)
            blank=False

    return "\n".join(cleaned)

In [29]:
links=get_links()

for link in tqdm(links):

    md=download_markdown(link)

    if md:

        md=clean_markdown(md)

        filename=link.rstrip("/").split("/")[-1]

        if filename=="":
            filename="index"

        path=os.path.join(DOCS_FOLDER,filename+".md")

        with open(path,"w",encoding="utf-8") as f:
            f.write(md)

print("Download Complete")

100%|██████████| 37/37 [00:15<00:00,  2.38it/s]

Download Complete
